In [6]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Import all necessary libraries
import pandas as pd 
import re

In [3]:
# import data 
df_a = pd.read_excel("Part A with Supplier.xlsx", sheet_name="Export Worksheet")
df_b = pd.read_excel("Part B with Supplier.xlsx", sheet_name="Export Worksheet")

c:\Users\ITafr\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


KeyboardInterrupt: 

In [ ]:
frames = [df_a, df_b]
df = pd.concat(frames, ignore_index=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1376677 entries, 0 to 1376676
Data columns (total 29 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   PART_NO           1376677 non-null  object 
 1   DESCRIPTION       1376677 non-null  object 
 2   MFR               1203435 non-null  object 
 3   STYLE             428364 non-null   object 
 4   COMPOSITION       372517 non-null   object 
 5   WIDTH             112200 non-null   object 
 6   COLOR             616087 non-null   object 
 7   TYPE              643611 non-null   object 
 8   SIZE              540604 non-null   object 
 9   GAUGE             335557 non-null   object 
 10  FACE WEIGHT       168144 non-null   object 
 11  COLLECTION        136996 non-null   object 
 12  DEPTH             31424 non-null    object 
 13  BACKING           281935 non-null   object 
 14  FINISH            142852 non-null   object 
 15  SITE              1376677 non-null  object 
 16  

In [ ]:
# How many null entries in part number
num_null_part_numbers = df['PART_NO'].isnull().sum()
print(f"Number of null entries in PART_NO: {num_null_part_numbers}")

Number of null entries in PART_NO: 0


In [ ]:
# Unique descriptions
num_unique_descriptions = df['DESCRIPTION'].nunique()
print(f"Number of unique descriptions: {num_unique_descriptions}")

Number of unique descriptions: 399469


In [ ]:
# Unique parts
num_unique_part = df['PART_NO'].nunique()
print(f"Number of unique Part numbers: {num_unique_descriptions}")

Number of unique Part numbers: 399469


Observation: <br>
    1) There are 398 609 unique parts <br>
    2) all part descriptions have unique part numbers 

In [ ]:
# Return a DataFrame with unique descriptions
unique_df = df.drop_duplicates(subset=['DESCRIPTION'])
unique_df = unique_df.reset_index(drop=True)
unique_df.head(2)

,PART_NO,DESCRIPTION,MFR,STYLE,COMPOSITION,WIDTH,COLOR,TYPE,SIZE,GAUGE,...,COMMODITY 2,PRODUCT CODE,PRODUCT_FAMILY,INV UoM,PURCH UoM,CONV FACTOR,PRIMARY SUPPLIER,SUPPLIER NAME,NET WEIGHT,NET VOLUME
0,1000042658,Miscellaneous Material - CTN,OTHR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2MISC,*,*,CTN,CTN,1.0,999999,SUPPLIER PLACEHOLDER,NaN,NaN
1,1000017103,Vortex 400g Orange,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2VCT,*,*,EA,EA,1.0,102549,"Aramsco, Inc.",NaN,NaN


In [ ]:
# Miscellaneous descriptions
num_miscellaneous_descriptions = unique_df['DESCRIPTION'].str.contains('Miscellaneous', case=False, na=False).sum()
print(f"Number of descriptions with 'Miscellaneous': {num_miscellaneous_descriptions}")

Number of descriptions with 'Miscellaneous': 23


In [ ]:
# Create a boolean mask for PART_NO in unique_df_clean that are also in df_miscellaneous
df_miscellaneous = unique_df[unique_df['DESCRIPTION'].str.contains('Miscellaneous', case=False, na=False)]
unique_df['miscellaneous'] = unique_df['PART_NO'].isin(df_miscellaneous['PART_NO'])

In [ ]:
# Filter out rows where DESCRIPTION starts with "Material -" (case-sensitive) ---
df_material = unique_df[unique_df['DESCRIPTION'].str.startswith("Material -", na=False)]
df_material.head(2)

,PART_NO,DESCRIPTION,MFR,STYLE,COMPOSITION,WIDTH,COLOR,TYPE,SIZE,GAUGE,...,PRODUCT CODE,PRODUCT_FAMILY,INV UoM,PURCH UoM,CONV FACTOR,PRIMARY SUPPLIER,SUPPLIER NAME,NET WEIGHT,NET VOLUME,miscellaneous
9,1000002136,Material - Non-Carpet Glued,OTHR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,EA,EA,1.0,1070,"Noble Flooring, LLC",NaN,NaN,False
10,1000002140,Material - Sundry Carpet Glued Related,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,EA,EA,1.0,SUB,Sub contractor Place holder,NaN,NaN,False


In [ ]:
# Create a boolean mask for PART_NO in unique_df_clean that are also in df_material
unique_df['material_part'] = unique_df['PART_NO'].isin(df_material['PART_NO'])

In [ ]:
# # Lets get the manufacture extraction function (extract the first 1 to 3 capitalized words)
# def extract_manufacture(description):
#     try:
#         # Match first 1 to 3 words starting with uppercase letters
#         match = re.match(r'^([A-Z][a-zA-Z]*(?:\s+[A-Z][a-zA-Z]*){0,2})', str(description))
#         return match.group(1).strip() if match else None
#     except:
#         return None

In [ ]:
# # Apply supplier extraction to the DESCRIPTION column
# unique_df_clean['Manufacture'] = unique_df_clean['DESCRIPTION'].apply(extract_manufacture)
# unique_df_clean[['DESCRIPTION', 'Manufacture']]


,DESCRIPTION,SUPPLIER


In [ ]:
# # --- Filter out rows where DESCRIPTION starts with "Material" (case-sensitive) ---
# unique_df_clean = df[~df['DESCRIPTION'].str.startswith("Material", na=False)]

### Lets bite the pie one manufacure at a time <br> 
1) First we need to separate this big data by manufacture. <br> 
2) We need to filter by commodity. <br> 
3) We need to create a functions to extract all the relevant information. 

In [ ]:
df_interface = unique_df[unique_df['DESCRIPTION'].str.contains(r'^interface', case=False, na=False, regex=True)]
df_interface.info()

NameError: name 'unique_df' is not defined

In [ ]:
# create a function to extract the relevant information from the description

# Regex for standard format

def extract_description_flexible(desc):
    if not isinstance(desc, str):
        return {}

    desc = desc.strip()

    # Pattern 1: Parentheses-style detailed descriptions
    pattern_1 = re.compile(r"""^Interface\s+
                                (?P<name>[^\(]+?)\s*
                                (?:\((?P<part_number>\d+)\))?\s+
                                (?P<product_type>\w+)\s+
                                (?P<form>(?:\w+\s*)+?)
                                (?P<dimensions>[\d\.]+\s*\w+\s+x\s+[\d\.]+\s*\w+)\s*
                                [-–]\s*
                                (?P<color>.+?)\s*
                                \((?P<color_code>[^)]+)\)
                                $""", re.VERBOSE | re.IGNORECASE)

    match_1 = pattern_1.match(desc)
    if match_1:
        return match_1.groupdict()

    # Pattern 2: Backslash-delimited format
    if '\\' in desc:
        parts = desc.split('\\')
        if len(parts) == 5:
            return {
                'collection': parts[1].strip(),
                'product_name': parts[2].strip(),
                'backing': parts[3].strip(),
                'dimensions': parts[4].strip(),
                'name': None,
                'part_number': None,
                'product_type': None,
                'form': None,
                'color': None,
                'color_code': None
            }

    # Pattern 3: No part number, but dash/en dash + color and color code
    pattern_3 = re.compile(r"""^Interface\s+
                                (?P<name>\w+)\s+
                                (?P<product_type>\w+)\s+
                                (?P<form>(?:\w+\s*)+?)
                                (?P<dimensions>[\d\.]+\s*\w+\s+x\s+[\d\.]+\s*\w+)\s*
                                [-–]\s*
                                (?P<color>.+?)\s*
                                \((?P<color_code>[^)]+)\)
                                $""", re.VERBOSE | re.IGNORECASE)

    match_3 = pattern_3.match(desc)
    if match_3:
        return match_3.groupdict()

    # Fallback if no match
    return {
        'collection': None,
        'product_name': None,
        'backing': None,
        'dimensions': None,
        'name': None,
        'part_number': None,
        'product_type': None,
        'form': None,
        'color': None,
        'color_code': None
    }

In [ ]:
extracted_df = df_interface['DESCRIPTION'].apply(extract_description_flexible).apply(pd.Series)
df_normalized = pd.concat([df_interface, extracted_df], axis=1)

NameError: name 'df_interface' is not defined

In [ ]:
df_normalized.to_excel("interface_normalized_v2.xlsx", index=False)